<a href="https://colab.research.google.com/github/1p2a3r4a/flyrank-project/blob/main/Copy_of_w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/1p2a3r4a/flyrank-project/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [ ]:
from google.colab import userdata
userdata.get('HF_TOKEN')
import duckdb
import pandas as pd

# ---------------------------------------------------------
# 1. LOAD DATA
# ---------------------------------------------------------
# Requires a Colab secret named HF_TOKEN (key icon in left sidebar),
# with "Notebook access" toggled ON.
HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

rel = "hf://datasets/FlyRank/internship-warehouse"
df = con.sql(f"""
SELECT *
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
LIMIT 100
""").df()

print("Columns:", df.columns.tolist())
print(df.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Columns: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']
  report_date           client_hash_id           content_hash_id  \
0  2025-01-27  client_9958f0a7ae1df715  content_3b70a18ea133b2bb   
1  2025-01-27  client_9958f0a7ae1df715  content_fe8e8155ce1d47a2   
2  2025-01-27  client_9958f0a7ae1df715  content_b4462a1b90640058   
3  2025-01-27  client_9958f0a7ae1df715  content_c899aef92518c714   
4  2025-01-27  client_9958f0a7ae1df715  content_c7c1d2e68d9d0964   

   client_has_gsc  client_has_ga4  gsc_data_ava

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [ ]:
print(df.info())
print(df.describe(include="all"))
df.head(10)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 31 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   report_date               100 non-null    datetime64[us]
 1   client_hash_id            100 non-null    object        
 2   content_hash_id           100 non-null    object        
 3   client_has_gsc            100 non-null    bool          
 4   client_has_ga4            100 non-null    bool          
 5   gsc_data_available        100 non-null    bool          
 6   ga4_data_available        100 non-null    bool          
 7   gsc_impressions           100 non-null    int64         
 8   gsc_clicks                100 non-null    int64         
 9   gsc_sum_position          100 non-null    int64         
 10  gsc_avg_position          100 non-null    float64       
 11  ga4_pageviews             100 non-null    int64         
 12  ga4_sessions           

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,2025-01
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,2025-01
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,2025-01
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,2025-01
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,2025-01
5,2025-01-27,client_9958f0a7ae1df715,content_c782fa8abd4fce5e,True,True,True,False,21,0,1050,...,0,0,0,0,0,0,0,0,0,2025-01
6,2025-01-27,client_9958f0a7ae1df715,content_ae5e5fd6edff550f,True,True,True,False,13,0,127,...,0,0,0,0,0,0,0,0,0,2025-01
7,2025-01-27,client_9958f0a7ae1df715,content_a64143f6e4a21ffe,True,True,True,False,29,0,356,...,0,0,0,0,0,0,0,0,0,2025-01
8,2025-01-27,client_9958f0a7ae1df715,content_e281674658070602,True,True,True,False,5,0,103,...,0,0,0,0,0,0,0,0,0,2025-01
9,2025-01-27,client_9958f0a7ae1df715,content_658f53fa439c66ca,True,True,True,False,8,0,304,...,0,0,0,0,0,0,0,0,0,2025-01


## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [ ]:
TARGET = "YOUR_TARGET_COLUMN"  # <-- replace with an actual column from df.columns.tolist()

if TARGET not in df.columns:
    numeric_cols = df.select_dtypes(include="number").columns.tolist()
    if not numeric_cols:
        raise ValueError("No numeric columns found in df to use as a target.")
    TARGET = numeric_cols[0]
    print(f"WARNING: TARGET not found in df.columns — auto-selected '{TARGET}' instead. "
          f"Edit TARGET manually if this isn't the column you want.")

X = df.drop(columns=[TARGET])
y = df[TARGET]

print("Target:", TARGET)
print("Features shape:", X.shape)
print("Target shape:", y.shape)

Target: gsc_impressions
Features shape: (100, 30)
Target shape: (100,)


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

# Separate datetime columns (they break numeric imputation/scaling)
datetime_cols = X.select_dtypes(include=["datetime64[ns]", "datetime64[ns, UTC]"]).columns
categorical = X.select_dtypes(include="object").columns
numeric = X.select_dtypes(include=["number"]).columns

print("Datetime columns (dropped):", list(datetime_cols))
print("Categorical columns:", list(categorical))
print("Numeric columns:", list(numeric))

X_model = X.drop(columns=datetime_cols)

preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric),
    ("cat",
     Pipeline([
         ("imputer", SimpleImputer(strategy="most_frequent")),
         ("encoder", OneHotEncoder(handle_unknown="ignore"))
     ]),
     categorical)
])

model = Pipeline([
    ("prep", preprocessor),
    ("model", LinearRegression())
])

X_train, X_test, y_train, y_test = train_test_split(
    X_model, y, test_size=0.2, random_state=42
)

model.fit(X_train, y_train)
pred = model.predict(X_test)
mae = mean_absolute_error(y_test, pred)

print("MAE:", mae)

Datetime columns (dropped): ['report_date']
Categorical columns: ['client_hash_id', 'content_hash_id', 'month']
Numeric columns: ['gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']
MAE: 15.352684566041168


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [ ]:
numeric_df = df.select_dtypes(include="number")
corr = numeric_df.corr()
print(corr)
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Target:", TARGET)
print("MAE:", mae)
print(df.columns.tolist())

                          gsc_impressions  gsc_clicks  gsc_sum_position  \
gsc_impressions                  1.000000    0.071872          0.450393   
gsc_clicks                       0.071872    1.000000         -0.036889   
gsc_sum_position                 0.450393   -0.036889          1.000000   
gsc_avg_position                -0.054230   -0.071534          0.637543   
ga4_pageviews                         NaN         NaN               NaN   
ga4_sessions                          NaN         NaN               NaN   
ga4_users                             NaN         NaN               NaN   
ga4_engaged_sessions                  NaN         NaN               NaN   
ga4_total_engagement_sec              NaN         NaN               NaN   
sessions_organic                      NaN         NaN               NaN   
sessions_direct                       NaN         NaN               NaN   
sessions_referral                     NaN         NaN               NaN   
sessions_social          

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.